# Lezione 11: Data warehouse, data mart, data lake, ETL e integrazione

**Obiettivi di apprendimento:**
- Distinguere warehouse, mart e lake.
- Descrivere i tre passi dell'ETL.
- Costruire una mini pipeline ETL con pandas e sqlite3.
- Spiegare cos'è l'integrazione dei dati e i metadati.

**Durata stimata:** 110 min


## Tre modi di centralizzare i dati

Immaginate un'azienda con dati distribuiti in venti sistemi diversi. Se ogni area analizza i propri dati separatamente, prima o poi due report si contraddiranno. Per questo esistono questi tre concetti:

- **Data warehouse**: repository centrale, già modellato e pronto per l'analisi — la "fonte unica di verità".
- **Data mart**: una porzione di quel warehouse per un'area di business specifica (vendite, finanza...), con sicurezza e prestazioni isolate.
- **Data lake**: archivio in formato nativo (grezzo) di qualsiasi tipo di dato, etichettato con metadati, senza escludere nulla — utile per l'analitica avanzata o come zona di preparazione (staging) prima di un warehouse.

## ETL: Extract, Transform, Load

Il processo che porta i dati dalla loro origine al warehouse ha tre passi:

1. **Extract**: tirare fuori i dati dalla loro fonte (a lotti, con strumenti come Stitch o Blendo; oppure in streaming, con Kafka, Storm o Samza).
2. **Transform**: standardizzare i formati, eliminare duplicati, filtrare, arricchire, dividere campi, applicare regole di business.
3. **Load**: caricare il risultato nella sua destinazione (caricamento iniziale, incrementale o refresh completo), verificando che non ci siano nulli inaspettati né fallimenti silenziosi.

Costruiamo i tre passi, dal vivo, con dati che imitano i problemi tipici dei dati reali.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

# EXTRACT: dati "grezzi" simulati, con problemi tipici —
# una riga duplicata, un cliente senza nome, un importo vuoto,
# maiuscole/minuscole incoerenti.
datos_crudos = pd.DataFrame({
    "id_venta":  [1, 2, 3, 4, 4, 5],
    "cliente":   ["Ana Torres", "luis peña", "CARLA RUIZ", "Diego Soto", "Diego Soto", None],
    "producto":  ["Camiseta", "camiseta", "Auriculares", None, None, "Mochila"],
    "importe":   [15.99, 15.99, 39.90, 25.00, 25.00, np.nan],
})

print("Dati grezzi:")
print(datos_crudos)

In [ ]:
# TRANSFORM: puliamo passo dopo passo
datos_limpios = datos_crudos.drop_duplicates()
datos_limpios = datos_limpios.dropna(subset=["cliente"])
datos_limpios["importe"] = datos_limpios["importe"].fillna(0)
datos_limpios["cliente"] = datos_limpios["cliente"].str.title()
datos_limpios["producto"] = datos_limpios["producto"].fillna("Sin especificar").str.title()
datos_limpios["importe"] = pd.to_numeric(datos_limpios["importe"], errors="coerce").fillna(0)
datos_limpios = datos_limpios[datos_limpios["importe"] >= 0]

print("Dati trasformati:")
print(datos_limpios)

In [ ]:
# LOAD: carichiamo il risultato pulito in una tabella relazionale,
# il nostro mini-warehouse in memoria.
conexion = sqlite3.connect(':memory:')
datos_limpios.to_sql("ventas", conexion, index=False, if_exists="replace")

verificacion = pd.read_sql_query("SELECT * FROM ventas", conexion)
print("Dati caricati nella tabella 'ventas':")
print(verificacion)

conexion.close()

## Oltre l'ETL: pipeline, integrazione e metadati

- Una **pipeline di dati** è più ampia dell'ETL: include batch e streaming, e non sempre termina in un warehouse (strumenti: Apache Beam, DataFlow). Ogni ETL è una pipeline, ma non ogni pipeline è un ETL.
- L'**integrazione dei dati** (secondo Gartner) è "le pratiche, le tecniche architetturali e gli strumenti che permettono alle organizzazioni di ingerire, trasformare, combinare e fornire dati attraverso diversi tipi di dati" — un concetto ancora più ampio, che include la coerenza tra applicazioni, la gestione dei dati anagrafici e le migrazioni. Strumenti reali: IBM Information Server, Talend, Informatica, tra gli altri.
- I **metadati** ("dati sui dati") sono di tre tipi: **tecnici** (struttura: tabelle, colonne), **di processo** (come sono stati spostati i dati: tempi, origine/destinazione) e **di business** (cosa significano i dati per l'organizzazione). Si organizzano in un **catalogo dati**, fondamentale per la governance e la provenienza dei dati (rintracciare un errore fino alla sua causa radice).

## Riepilogo della lezione

- Warehouse = centralizzato e modellato; mart = porzione per area di business; lake = grezzo, non trasformato.
- ETL = Extract + Transform + Load; una pipeline di dati è più ampia; l'integrazione dei dati è ancora più ampia.
- `pandas` (`drop_duplicates`, `dropna`, `fillna`, normalizzazione del testo) + `sqlite3` (`to_sql`) permettono di costruire un vero ETL, anche se piccolo.
- I metadati tecnici, di processo e di business vivono in un catalogo dati, base della governance e della provenienza dei dati.

**Capitolo del manuale:** [13 — Warehouse, mart, lake, ETL e pipeline](../manual/13_warehouses_etl_pipelines_IT.md)

**Notebook di pratica:** [clase11_practica_warehouses_etl_pipelines_IT.ipynb](../notebooks_practica/clase11_practica_warehouses_etl_pipelines_IT.ipynb)